# Proposta Marcelo - Leitura de dados

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcelo7bastos/relatorio_dados_damei/blob/main/notebooks/proposta_marcelo.ipynb)

Este notebook inicia o fluxo do projeto. O objetivo desta etapa é configurar a origem dos dados, listar as bases disponíveis e confirmar que conseguimos ler os arquivos Excel necessários para o relatório estadual.

## 1. Importar bibliotecas

Usamos bibliotecas padrão do Python para caminhos, datas e ambiente, além de `pandas` para ler as planilhas Excel.

In [57]:
from datetime import datetime
from pathlib import Path
import os

import pandas as pd


## 2. Configurar origem dos dados e pasta de saída

O GitHub guarda código, documentação e template. O Google Drive é o repositório operacional dos dados e dos relatórios gerados.

Existem apenas dois modos:

- `local`: roda no VS Code e usa uma cópia local/mock em `dados_brutos/dado_atual`.
- `google_drive`: roda no Google Colab, monta o Google Drive e lê os dados oficiais do Drive.

In [58]:
# O modo é escolhido automaticamente:
# - Google Colab: "google_drive"
# - VS Code/local: "local"
MODO_DADOS = "google_drive" if "COLAB_RELEASE_TAG" in os.environ else "local"

# UF usada nas próximas etapas do projeto.
UF_INTERESSE = "MG"

# Caminhos no Google Drive para execução no Colab.
GOOGLE_DRIVE_RAW_DIR = Path("/content/drive/MyDrive/MDA/DAMEI_Relatorio_Dados/dados_brutos/dado_atual")
GOOGLE_DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/MDA/dado_atual")

# Subpastas usadas somente no modo local.
RAW_SUBDIR = Path("dados_brutos") / "dado_atual"
OUTPUT_SUBDIR = Path("relatorios_gerados")

# Data de execução usada futuramente no nome dos arquivos gerados.
RUN_TIMESTAMP = datetime.now()
RUN_YYYYMM = RUN_TIMESTAMP.strftime("%Y%m")
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")


## 3. Resolver caminhos

Esta célula transforma a configuração acima em caminhos finais de leitura e gravação.

No modo `local`, o notebook usa as pastas do próprio repositório. No modo `google_drive`, o notebook exige Google Colab, monta o Drive e usa os caminhos `/content/drive/...`.

In [59]:
def esta_no_colab() -> bool:
    """Retorna True quando o notebook está rodando no Google Colab."""
    return "COLAB_RELEASE_TAG" in os.environ


def encontrar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Procura a raiz do projeto apenas no modo local."""
    inicio = (inicio or Path.cwd()).resolve()

    for candidato in [inicio, *inicio.parents]:
        tem_notebooks = (candidato / "notebooks").exists()
        tem_requirements = (candidato / "requirements.txt").exists()
        if tem_notebooks and tem_requirements:
            return candidato

    raise FileNotFoundError(
        "Não encontrei a raiz do projeto para o modo local. "
        "Abra o VS Code na pasta relatorio_dados_damei ou execute o notebook "
        "a partir de uma subpasta do repositório."
    )


if MODO_DADOS == "local":
    PROJECT_ROOT = encontrar_raiz_projeto()
    DATA_PROJECT_ROOT = PROJECT_ROOT
    RAW_DIR = PROJECT_ROOT / RAW_SUBDIR
    OUTPUT_BASE_DIR = PROJECT_ROOT / OUTPUT_SUBDIR
    OUTPUT_DIR = OUTPUT_BASE_DIR / RUN_YYYYMM
elif MODO_DADOS == "google_drive":
    if not esta_no_colab():
        raise RuntimeError(
            'MODO_DADOS = "google_drive" só deve ser usado no Google Colab. '
            'No VS Code local, use MODO_DADOS = "local".'
        )

    from google.colab import drive

    drive.mount("/content/drive")
    DATA_PROJECT_ROOT = GOOGLE_DRIVE_RAW_DIR.parents[1]
    RAW_DIR = GOOGLE_DRIVE_RAW_DIR
    OUTPUT_DIR = GOOGLE_DRIVE_OUTPUT_DIR
else:
    raise ValueError('MODO_DADOS deve ser "local" ou "google_drive".')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Modo de dados:     {MODO_DADOS}")
print(f"UF de interesse:   {UF_INTERESSE}")
print(f"Projeto de dados:  {DATA_PROJECT_ROOT}")
print(f"Pasta de dados:    {RAW_DIR}")
print(f"Pasta de saída:    {OUTPUT_DIR}")


Modo de dados:     local
UF de interesse:   MG
Projeto de dados:  C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei
Pasta de dados:    C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual
Pasta de saída:    C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\relatorios_gerados\202604


## 4. Validar pastas de dados e saída

Antes de ler qualquer arquivo, validamos se a pasta de dados existe. A pasta de saída é criada automaticamente quando ainda não existir. No modo `google_drive`, ela usa o caminho configurado em `GOOGLE_DRIVE_OUTPUT_DIR`.

In [60]:
if not RAW_DIR.exists():
    raise FileNotFoundError(f"A pasta de dados não existe: {RAW_DIR}")

print("Pasta de dados encontrada.")
print("Pasta de saída pronta.")


Pasta de dados encontrada.
Pasta de saída pronta.


## 5. Listar arquivos Excel

Aqui verificamos quais arquivos `.xlsx` estão disponíveis na pasta de dados configurada.

In [61]:
excel_files = sorted(RAW_DIR.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em {RAW_DIR}")

print(f"Foram encontrados {len(excel_files)} arquivos Excel:")
for arquivo in excel_files:
    tamanho_mb = arquivo.stat().st_size / (1024 * 1024)
    print(f"- {arquivo.name} ({tamanho_mb:.2f} MB)")


Foram encontrados 7 arquivos Excel:
- ater_ate_2026_03_gerado_em_20260410151127.xlsx (0.03 MB)
- caf_dap_ativos_ate_2026_03_gerado_em_20260410152650.xlsx (0.31 MB)
- GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx (0.09 MB)
- mais_alimentos_gaia_202604151554.xlsx (0.22 MB)
- pncf_2026_03_gerado_em_20260410170006.xlsx (0.02 MB)
- PNRA_2026_2026_04_15.xlsx (0.29 MB)
- pronaf_gaia_20260414.xlsx (0.64 MB)


## 6. Identificar arquivos por política

Nesta etapa fazemos um primeiro mapeamento automático entre nomes de arquivos e políticas públicas. Essa lógica ainda é simples, mas já ajuda a validar se temos todas as bases esperadas.

In [62]:
arquivos_encontrados = {
    "caf_dap": None,
    "ater": None,
    "pronaf": None,
    "mais_alimentos": None,
    "pncf": None,
    "garantia_safra": None,
    "pnra": None,
}

for arquivo in excel_files:
    nome = arquivo.name.lower()

    if "caf" in nome:
        arquivos_encontrados["caf_dap"] = arquivo
    elif "ater" in nome:
        arquivos_encontrados["ater"] = arquivo
    elif "pronaf" in nome:
        arquivos_encontrados["pronaf"] = arquivo
    elif "mais_alimentos" in nome or "mais-alimentos" in nome:
        arquivos_encontrados["mais_alimentos"] = arquivo
    elif "pncf" in nome:
        arquivos_encontrados["pncf"] = arquivo
    elif "garantia-safra" in nome or "garantia_safra" in nome:
        arquivos_encontrados["garantia_safra"] = arquivo
    elif "pnra" in nome:
        arquivos_encontrados["pnra"] = arquivo

resumo_arquivos = pd.DataFrame(
    [
        {
            "politica": politica,
            "arquivo": arquivo.name if arquivo else "NÃO ENCONTRADO",
            "encontrado": arquivo is not None,
        }
        for politica, arquivo in arquivos_encontrados.items()
    ]
)

resumo_arquivos


,politica,arquivo,encontrado
0,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,True
1,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,True
2,pronaf,pronaf_gaia_20260414.xlsx,True
3,mais_alimentos,mais_alimentos_gaia_202604151554.xlsx,True
4,pncf,pncf_2026_03_gerado_em_20260410170006.xlsx,True
5,garantia_safra,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,True
6,pnra,PNRA_2026_2026_04_15.xlsx,True


## 7. Inspecionar abas e colunas

Agora abrimos cada arquivo com `pandas.ExcelFile`, listamos as abas e coletamos os nomes das colunas. Este teste ainda não transforma os dados; ele apenas confirma que os arquivos são legíveis.

In [63]:
linhas = []

for politica, arquivo in arquivos_encontrados.items():
    if arquivo is None:
        linhas.append(
            {
                "politica": politica,
                "arquivo": "NÃO ENCONTRADO",
                "aba": None,
                "qtd_colunas": None,
                "colunas": None,
            }
        )
        continue

    xls = pd.ExcelFile(arquivo)
    for aba in xls.sheet_names:
        colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0).columns.astype(str).tolist()
        linhas.append(
            {
                "politica": politica,
                "arquivo": arquivo.name,
                "aba": aba,
                "qtd_colunas": len(colunas),
                "colunas": ", ".join(colunas[:12]) + (" ..." if len(colunas) > 12 else ""),
            }
        )

df_estrutura = pd.DataFrame(linhas)
df_estrutura


,politica,arquivo,aba,qtd_colunas,colunas
0,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,Orientações,1,Orientações
1,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,DADOS,9,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
2,caf_dap,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,TOTALIZADORES,1,Planilha de Dados - Totalizadores
3,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,Orientações,1,Orientações
4,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,DADOS,7,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
5,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,TOTALIZADORES,1,Planilha de Dados - Totalizadores
6,pronaf,pronaf_gaia_20260414.xlsx,Resumo,1,Orientacoes gerais sobre a planilha de dados:
7,pronaf,pronaf_gaia_20260414.xlsx,Totalizadores,17,"dt_referencia, dt_geracao, uf, cod_ibge_uf, AN..."
8,pronaf,pronaf_gaia_20260414.xlsx,Dados,19,"dt_referencia, dt_geracao, ANO, nome_municipio..."
9,mais_alimentos,mais_alimentos_gaia_202604151554.xlsx,Resumo,1,Orientações gerais sobre a planilha de dados:


## 8. Testar leitura das abas principais

Por fim, carregamos as abas principais de cada política e mostramos o tamanho de cada tabela. Se esta célula rodar sem erro, o teste de leitura está aprovado.

In [64]:
abas_principais = {
    "caf_dap": "DADOS",
    "ater": "DADOS",
    "pronaf": "Dados",
    "mais_alimentos": "Dados",
    "pncf": "DADOS",
    "garantia_safra": "DADOS",
    "pnra": "DADOS",
}

tabelas = {}
resumo_leitura = []

for politica, aba in abas_principais.items():
    arquivo = arquivos_encontrados.get(politica)

    if arquivo is None:
        resumo_leitura.append(
            {
                "politica": politica,
                "status": "arquivo não encontrado",
                "linhas": None,
                "colunas": None,
            }
        )
        continue

    df = pd.read_excel(arquivo, sheet_name=aba)
    tabelas[politica] = df
    resumo_leitura.append(
        {
            "politica": politica,
            "status": "ok",
            "linhas": df.shape[0],
            "colunas": df.shape[1],
        }
    )

pd.DataFrame(resumo_leitura)


,politica,status,linhas,colunas
0,caf_dap,ok,5526,9
1,ater,ok,328,7
2,pronaf,ok,4796,19
3,mais_alimentos,ok,3935,8
4,pncf,ok,77,9
5,garantia_safra,ok,1215,9
6,pnra,ok,5600,10


## 9. Incorporação da proposta Wesley

A partir daqui incorporamos os avanços do notebook `proposta_wesley.ipynb`: uso dos totalizadores de CAF e PRONAF, comparação UF x Brasil e geração de um primeiro relatório Word de teste. A implementação abaixo usa os caminhos já configurados neste notebook, sem caminhos fixos e sem download automático.

## 10. Funções auxiliares

Estas funções padronizam validação, percentuais e formatação brasileira de números antes da geração do relatório.

In [65]:
try:
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH


def normalizar_uf(uf: str) -> str:
    return str(uf).strip().upper()


def validar_colunas(df: pd.DataFrame, colunas: list[str], contexto: str) -> None:
    faltantes = [col for col in colunas if col not in df.columns]
    if faltantes:
        raise KeyError(f"Colunas ausentes em {contexto}: {faltantes}")


def calcular_percentual(parte: float, total: float) -> float:
    if pd.isna(total) or total == 0:
        return 0.0
    return float(parte) / float(total) * 100


def formatar_inteiro(valor: float) -> str:
    return f"{valor:,.0f}".replace(",", ".")


def formatar_moeda(valor: float) -> str:
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_percentual(valor: float) -> str:
    return f"{valor:.2f}%".replace(".", ",")


## 11. Carregar Políticas

Nesta etapa carregamos e tratamos cada política separadamente. A regra é: localizar o arquivo esperado, ler a aba correta, validar as colunas mínimas e deixar uma tabela pronta para uso nos indicadores do relatório.

### Carregar CAF

Origem: `caf_dap_ativos_ate_2026_03_gerado_em_20260410152650.xlsx`. Usamos a aba `TOTALIZADORES`, que já traz os valores consolidados por UF.

In [66]:
arquivo_caf = arquivos_encontrados.get("caf_dap")
if arquivo_caf is None:
    raise FileNotFoundError("Arquivo CAF/DAP não encontrado.")

df_caf_totalizadores = pd.read_excel(
    arquivo_caf,
    sheet_name="TOTALIZADORES",
    engine="openpyxl",
    skiprows=6,
    nrows=27,
    na_values=["NA", "na", ""],
)
df_caf_totalizadores.columns = df_caf_totalizadores.columns.str.strip()

caf_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge_estados", "uf"]
caf_colunas_numericas = [
    "Soma de CAFs PF ATIVO",
    "Soma de CAFs PJ ATIVO",
    "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO",
    "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO",
]
validar_colunas(df_caf_totalizadores, caf_colunas_texto + caf_colunas_numericas, "CAF totalizadores")

for col in caf_colunas_texto:
    df_caf_totalizadores[col] = df_caf_totalizadores[col].astype(str).str.strip()
for col in caf_colunas_numericas:
    df_caf_totalizadores[col] = pd.to_numeric(df_caf_totalizadores[col], errors="coerce").fillna(0)

df_caf_totalizadores["uf"] = df_caf_totalizadores["uf"].str.upper()
df_caf_totalizadores.head()

,dt_referencia,dt_geracao,cod_ibge_estados,uf,Soma de CAFs PF ATIVO,Soma de CAFs PJ ATIVO,Soma de QUANTIDADE DE MULHERES EM CAF ATIVO,Soma de QUANTIDADE DE HOMENS EM CAF ATIVO
0,2026_03,2026_04_10,11,RO,58847,58,48726,57505
1,2026_03,2026_04_10,12,AC,33319,96,28344,32882
2,2026_03,2026_04_10,13,AM,65014,293,59394,64112
3,2026_03,2026_04_10,14,RR,13972,60,10593,11735
4,2026_03,2026_04_10,15,PA,216099,364,170963,180953


### Carregar PRONAF

Origem: `pronaf_gaia_20260414.xlsx`. Usamos a aba `Totalizadores`, com recortes por UF, ano e gênero.

In [67]:
arquivo_pronaf = arquivos_encontrados.get("pronaf")
if arquivo_pronaf is None:
    raise FileNotFoundError("Arquivo PRONAF não encontrado.")

df_pronaf_totalizadores = pd.read_excel(
    arquivo_pronaf,
    sheet_name="Totalizadores",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pronaf_totalizadores.columns = df_pronaf_totalizadores.columns.str.strip()

pronaf_colunas_texto = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf"]
pronaf_colunas_numericas = [
    col
    for col in df_pronaf_totalizadores.columns
    if col not in pronaf_colunas_texto
]
validar_colunas(df_pronaf_totalizadores, pronaf_colunas_texto, "PRONAF totalizadores")

for col in pronaf_colunas_texto:
    df_pronaf_totalizadores[col] = df_pronaf_totalizadores[col].astype(str).str.strip()
for col in pronaf_colunas_numericas:
    df_pronaf_totalizadores[col] = pd.to_numeric(df_pronaf_totalizadores[col], errors="coerce").fillna(0)

df_pronaf_totalizadores["uf"] = df_pronaf_totalizadores["uf"].str.upper()
df_pronaf_totalizadores.head()

,dt_referencia,dt_geracao,uf,cod_ibge_uf,ANO,qtd_contratos_1_Feminino,qtd_contratos_1_Masculino,qtd_contratos_1_Sem_Identificacao,valor_total_contratos_1_Feminino,valor_total_contratos_1_Masculino,valor_total_contratos_1_Sem_Identificacao,qtd_operacoes_1_Feminino,qtd_operacoes_1_Masculino,qtd_operacoes_1_Sem_Identificacao,ticket_medio_1_Feminino,ticket_medio_1_Masculino,ticket_medio_1_Sem_Identificacao
0,2026_03,2026_04_14,AC,12,2026,443,662,3142,2.592383e+07,4.933696e+07,7.690108e+07,443,662,3142,58518.802348,74527.134169,24475.202018
1,2026_03,2026_04_14,AL,27,2026,5135,5667,4049,6.315917e+07,8.581216e+07,7.829423e+07,5135,5667,4049,12299.740382,15142.431376,19336.682544
2,2026_03,2026_04_14,AM,13,2026,308,463,2166,4.576762e+06,8.619838e+06,3.181671e+07,308,463,2166,14859.617143,18617.360670,14689.152839
3,2026_03,2026_04_14,AP,16,2026,235,264,1737,5.476934e+06,5.774499e+06,2.435346e+07,235,264,1737,23306.101957,21873.103750,14020.414634
4,2026_03,2026_04_14,BA,29,2026,27852,28931,27465,2.991230e+08,5.084193e+08,4.221115e+08,27852,28931,27465,10739.731177,17573.513206,15369.070985


### Carregar Mais Alimentos

Origem: `mais_alimentos_gaia_202604151554.xlsx`. Usamos a aba `Totalizadores`, que já consolida quantidade e valor por UF.

In [68]:
arquivo_mais_alimentos = arquivos_encontrados.get("mais_alimentos")
if arquivo_mais_alimentos is None:
    raise FileNotFoundError("Arquivo Mais Alimentos não encontrado.")

df_mais_alimentos_totalizadores = pd.read_excel(
    arquivo_mais_alimentos,
    sheet_name="Totalizadores",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_mais_alimentos_totalizadores.columns = df_mais_alimentos_totalizadores.columns.str.strip()

mais_alimentos_colunas_texto = ["dt_referencia", "dt_geracao", "CD_ESTADO", "cod_ibge_uf"]
mais_alimentos_colunas_numericas = ["qtd_contratos", "valor_total_contratos"]
validar_colunas(
    df_mais_alimentos_totalizadores,
    mais_alimentos_colunas_texto + mais_alimentos_colunas_numericas,
    "Mais Alimentos totalizadores",
)

for col in mais_alimentos_colunas_texto:
    df_mais_alimentos_totalizadores[col] = df_mais_alimentos_totalizadores[col].astype(str).str.strip()
for col in mais_alimentos_colunas_numericas:
    df_mais_alimentos_totalizadores[col] = pd.to_numeric(df_mais_alimentos_totalizadores[col], errors="coerce").fillna(0)

df_mais_alimentos_totalizadores["uf"] = df_mais_alimentos_totalizadores["CD_ESTADO"].str.upper()
df_mais_alimentos_totalizadores.head()

,dt_referencia,dt_geracao,CD_ESTADO,cod_ibge_uf,qtd_contratos,valor_total_contratos,uf
0,2026_03,2026_04_14,AC,12,1339,1.983638e+07,AC
1,2026_03,2026_04_14,AL,27,2127,1.376088e+07,AL
2,2026_03,2026_04_14,AM,13,1631,1.995172e+07,AM
3,2026_03,2026_04_14,AP,16,1372,1.799990e+07,AP
4,2026_03,2026_04_14,BA,29,13068,1.474165e+08,BA


### Carregar ATER

Origem: `ater_ate_2026_03_gerado_em_20260410151127.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [69]:
arquivo_ater = arquivos_encontrados.get("ater")
if arquivo_ater is None:
    raise FileNotFoundError("Arquivo ATER não encontrado.")

df_ater_dados = pd.read_excel(
    arquivo_ater,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_ater_dados.columns = df_ater_dados.columns.str.strip()

ater_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]
ater_colunas_numericas = [
    "familias_com_ater_iniciada_no_ano",
    "familias_com_ater_recebida_no_ano",
]
validar_colunas(df_ater_dados, ater_colunas_texto + ater_colunas_numericas, "ATER dados")

for col in ater_colunas_texto:
    df_ater_dados[col] = df_ater_dados[col].astype(str).str.strip()
for col in ater_colunas_numericas:
    df_ater_dados[col] = pd.to_numeric(df_ater_dados[col], errors="coerce").fillna(0)

df_ater_dados["uf"] = df_ater_dados["uf"].str.upper()

df_ater_totalizadores = (
    df_ater_dados
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        familias_com_ater_iniciada_no_ano=("familias_com_ater_iniciada_no_ano", "sum"),
        familias_com_ater_recebida_no_ano=("familias_com_ater_recebida_no_ano", "sum"),
    )
)
df_ater_totalizadores.head()

,uf,dt_referencia,dt_geracao,familias_com_ater_iniciada_no_ano,familias_com_ater_recebida_no_ano
0,AL,2026_03,20260410,143,146
1,AM,2026_03,20260410,0,130
2,AP,2026_03,20260410,396,714
3,BA,2026_03,20260410,141,646
4,CE,2026_03,20260410,1,415


### Carregar Garantia-Safra

Origem: `GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [70]:
arquivo_garantia_safra = arquivos_encontrados.get("garantia_safra")
if arquivo_garantia_safra is None:
    raise FileNotFoundError("Arquivo Garantia-Safra não encontrado.")

df_garantia_safra_dados = pd.read_excel(
    arquivo_garantia_safra,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_garantia_safra_dados.columns = df_garantia_safra_dados.columns.str.strip()

garantia_safra_colunas_texto = [
    "dt_referencia",
    "dt_geracao",
    "uf",
    "cod_ibge_uf",
    "cod_ibge",
    "nome_municipio",
    "pagamento_liberado",
]
garantia_safra_colunas_numericas = [
    "Agricultores_aderidos",
    "Agricultores_com_pagamento liberado",
]
validar_colunas(
    df_garantia_safra_dados,
    garantia_safra_colunas_texto + garantia_safra_colunas_numericas,
    "Garantia-Safra dados",
)

for col in garantia_safra_colunas_texto:
    df_garantia_safra_dados[col] = df_garantia_safra_dados[col].astype(str).str.strip()
for col in garantia_safra_colunas_numericas:
    df_garantia_safra_dados[col] = pd.to_numeric(df_garantia_safra_dados[col], errors="coerce").fillna(0)

df_garantia_safra_dados["uf"] = df_garantia_safra_dados["uf"].str.upper()

df_garantia_safra_totalizadores = (
    df_garantia_safra_dados
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        agricultores_aderidos=("Agricultores_aderidos", "sum"),
        agricultores_com_pagamento_liberado=("Agricultores_com_pagamento liberado", "sum"),
    )
)
df_garantia_safra_totalizadores.head()

,uf,dt_referencia,dt_geracao,agricultores_aderidos,agricultores_com_pagamento_liberado
0,AL,2023-2024,2026_04_13,28161,24814
1,AM,2023-2024,2026_04_13,705,274
2,BA,2023-2024,2026_04_13,333616,317522
3,CE,2023-2024,2026_04_13,176528,146296
4,MA,2023-2024,2026_04_13,2530,1362


### Carregar PNCF

Origem: `pncf_2026_03_gerado_em_20260410170006.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [71]:
import re
import unicodedata

arquivo_pncf = arquivos_encontrados.get("pncf")
if arquivo_pncf is None:
    raise FileNotFoundError("Arquivo PNCF não encontrado.")

def normalizar_nome_coluna(coluna: object) -> str:
    texto = unicodedata.normalize("NFKD", str(coluna))
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = texto.lower().replace("º", "o").replace("°", "o")
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")

df_pncf_dados = pd.read_excel(
    arquivo_pncf,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pncf_dados.columns = df_pncf_dados.columns.str.strip()

pncf_mapa_colunas = {
    "municipio": "municipio",
    "no_de_operacoes": "numero_operacoes",
    "valor_liberado_r": "valor_liberado",
    "valor_medio_liberado_r_operacao": "valor_medio_liberado",
}
df_pncf_dados = df_pncf_dados.rename(
    columns={
        coluna: pncf_mapa_colunas.get(normalizar_nome_coluna(coluna), coluna)
        for coluna in df_pncf_dados.columns
    }
)

pncf_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge", "sigla_uf", "nome_uf", "municipio"]
pncf_colunas_numericas = ["numero_operacoes", "valor_liberado", "valor_medio_liberado"]
validar_colunas(df_pncf_dados, pncf_colunas_texto + pncf_colunas_numericas, "PNCF dados")

for col in pncf_colunas_texto:
    df_pncf_dados[col] = df_pncf_dados[col].astype(str).str.strip()
for col in pncf_colunas_numericas:
    df_pncf_dados[col] = pd.to_numeric(df_pncf_dados[col], errors="coerce").fillna(0)

df_pncf_dados["sigla_uf"] = df_pncf_dados["sigla_uf"].str.upper()

df_pncf_totalizadores = (
    df_pncf_dados
    .groupby("sigla_uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        numero_operacoes=("numero_operacoes", "sum"),
        valor_liberado=("valor_liberado", "sum"),
    )
    .rename(columns={"sigla_uf": "uf"})
)
df_pncf_totalizadores["valor_medio_liberado"] = df_pncf_totalizadores.apply(
    lambda row: row["valor_liberado"] / row["numero_operacoes"] if row["numero_operacoes"] else 0,
    axis=1,
)

df_pncf_totalizadores.head()

,uf,dt_referencia,dt_geracao,numero_operacoes,valor_liberado,valor_medio_liberado
0,AL,2026_03,2026_04_10,10,2532386.00,253238.600000
1,BA,2026_03,2026_04_10,40,10535558.58,263388.964500
2,CE,2026_03,2026_04_10,42,7974624.24,189872.005714
3,ES,2026_03,2026_04_10,4,828048.00,207012.000000
4,GO,2026_03,2026_04_10,11,3093500.00,281227.272727


### Carregar PNRA

Origem: `PNRA_2026_2026_04_15.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [72]:
arquivo_pnra = arquivos_encontrados.get("pnra")
if arquivo_pnra is None:
    raise FileNotFoundError("Arquivo PNRA não encontrado.")

df_pnra_dados = pd.read_excel(
    arquivo_pnra,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pnra_dados.columns = df_pnra_dados.columns.str.strip()

pnra_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]
pnra_colunas_numericas = [
    "numero_familias_pa_criado_diferenciado_pnra",
    "numero_familias_pa_criado_tradicional_pnra",
    "numero_familias_reconhecimento_pnra",
    "numero_familias_regularizacao_pnra",
    "total_numero_familias_pnra",
]
validar_colunas(df_pnra_dados, pnra_colunas_texto + pnra_colunas_numericas, "PNRA dados")

for col in pnra_colunas_texto:
    df_pnra_dados[col] = df_pnra_dados[col].astype(str).str.strip()
for col in pnra_colunas_numericas:
    df_pnra_dados[col] = pd.to_numeric(df_pnra_dados[col], errors="coerce").fillna(0)

df_pnra_dados["uf"] = df_pnra_dados["uf"].str.upper()

df_pnra_totalizadores = (
    df_pnra_dados
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        numero_familias_pa_criado_diferenciado_pnra=("numero_familias_pa_criado_diferenciado_pnra", "sum"),
        numero_familias_pa_criado_tradicional_pnra=("numero_familias_pa_criado_tradicional_pnra", "sum"),
        numero_familias_reconhecimento_pnra=("numero_familias_reconhecimento_pnra", "sum"),
        numero_familias_regularizacao_pnra=("numero_familias_regularizacao_pnra", "sum"),
        total_numero_familias_pnra=("total_numero_familias_pnra", "sum"),
    )
)
df_pnra_totalizadores.head()

,uf,dt_referencia,dt_geracao,numero_familias_pa_criado_diferenciado_pnra,numero_familias_pa_criado_tradicional_pnra,numero_familias_reconhecimento_pnra,numero_familias_regularizacao_pnra,total_numero_familias_pnra
0,AC,2026 até mar,2026_04_15,281.0,0.0,20.0,170.0,471.0
1,AL,2026 até mar,2026_04_15,0.0,0.0,5.0,17.0,22.0
2,AM,2026 até mar,2026_04_15,498.0,0.0,0.0,130.0,628.0
3,AP,2026 até mar,2026_04_15,30.0,0.0,0.0,12.0,42.0
4,BA,2026 até mar,2026_04_15,0.0,0.0,0.0,972.0,972.0


### Resumo das políticas carregadas

In [73]:
politicas_tratadas = {
    "CAF": df_caf_totalizadores,
    "PRONAF": df_pronaf_totalizadores,
    "Mais Alimentos": df_mais_alimentos_totalizadores,
    "ATER": df_ater_totalizadores,
    "Garantia-Safra": df_garantia_safra_totalizadores,
    "PNCF": df_pncf_totalizadores,
    "PNRA": df_pnra_totalizadores,
}

resumo_politicas = pd.DataFrame(
    [
        {
            "politica": nome,
            "linhas": len(df),
            "colunas": len(df.columns),
            "ufs": df["uf"].nunique() if "uf" in df.columns else None,
        }
        for nome, df in politicas_tratadas.items()
    ]
)
resumo_politicas

,politica,linhas,colunas,ufs
0,CAF,27,8,27
1,PRONAF,27,17,27
2,Mais Alimentos,28,7,28
3,ATER,24,5,24
4,Garantia-Safra,11,5,11
5,PNCF,17,6,17
6,PNRA,28,8,28


## 12. Calcular indicadores UF x Brasil

Aqui consolidamos os principais indicadores de todas as políticas carregadas. O resultado é uma tabela única, com valor da UF escolhida, valor Brasil e participação percentual da UF.

In [74]:
UF_INTERESSE = normalizar_uf(UF_INTERESSE)


def filtrar_uf(df: pd.DataFrame, politica: str) -> pd.DataFrame:
    validar_colunas(df, ["uf"], politica)
    df_uf = df[df["uf"].astype(str).str.upper() == UF_INTERESSE].copy()
    if df_uf.empty:
        raise ValueError(f"UF {UF_INTERESSE} não encontrada nos dados de {politica}.")
    return df_uf


def somar_coluna(df: pd.DataFrame, coluna: str, contexto: str) -> float:
    validar_colunas(df, [coluna], contexto)
    return pd.to_numeric(df[coluna], errors="coerce").fillna(0).sum()


def somar_colunas(df: pd.DataFrame, colunas: list[str], contexto: str) -> float:
    validar_colunas(df, colunas, contexto)
    return sum(somar_coluna(df, coluna, contexto) for coluna in colunas)


def dividir_com_seguranca(numerador: float, denominador: float) -> float:
    if pd.isna(denominador) or denominador == 0:
        return 0.0
    return float(numerador) / float(denominador)


def adicionar_indicador(
    linhas: list[dict],
    politica: str,
    indicador: str,
    valor_uf: float,
    valor_brasil: float,
    formato: str = "inteiro",
) -> None:
    linhas.append(
        {
            "politica": politica,
            "indicador": indicador,
            "uf": valor_uf,
            "brasil": valor_brasil,
            "percentual_uf_br": calcular_percentual(valor_uf, valor_brasil),
            "formato": formato,
        }
    )


linhas_politicas = []

# CAF
for indicador, coluna in [
    ("CAFs Pessoa Física ativos", "Soma de CAFs PF ATIVO"),
    ("CAFs Pessoa Jurídica ativos", "Soma de CAFs PJ ATIVO"),
    ("Mulheres em CAF ativo", "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO"),
    ("Homens em CAF ativo", "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO"),
]:
    df_uf = filtrar_uf(df_caf_totalizadores, "CAF")
    adicionar_indicador(
        linhas_politicas,
        "CAF",
        indicador,
        somar_coluna(df_uf, coluna, "CAF UF"),
        somar_coluna(df_caf_totalizadores, coluna, "CAF Brasil"),
    )

# PRONAF
pronaf_colunas_qtd_contratos = [
    "qtd_contratos_1_Feminino",
    "qtd_contratos_1_Masculino",
    "qtd_contratos_1_Sem_Identificacao",
]
pronaf_colunas_valor_contratos = [
    "valor_total_contratos_1_Feminino",
    "valor_total_contratos_1_Masculino",
    "valor_total_contratos_1_Sem_Identificacao",
]
pronaf_colunas_qtd_operacoes = [
    "qtd_operacoes_1_Feminino",
    "qtd_operacoes_1_Masculino",
    "qtd_operacoes_1_Sem_Identificacao",
]
df_pronaf_uf = filtrar_uf(df_pronaf_totalizadores, "PRONAF")
adicionar_indicador(
    linhas_politicas,
    "PRONAF",
    "Contratos",
    somar_colunas(df_pronaf_uf, pronaf_colunas_qtd_contratos, "PRONAF UF"),
    somar_colunas(df_pronaf_totalizadores, pronaf_colunas_qtd_contratos, "PRONAF Brasil"),
)
adicionar_indicador(
    linhas_politicas,
    "PRONAF",
    "Valor total dos contratos",
    somar_colunas(df_pronaf_uf, pronaf_colunas_valor_contratos, "PRONAF UF"),
    somar_colunas(df_pronaf_totalizadores, pronaf_colunas_valor_contratos, "PRONAF Brasil"),
    formato="moeda",
)
adicionar_indicador(
    linhas_politicas,
    "PRONAF",
    "Operações",
    somar_colunas(df_pronaf_uf, pronaf_colunas_qtd_operacoes, "PRONAF UF"),
    somar_colunas(df_pronaf_totalizadores, pronaf_colunas_qtd_operacoes, "PRONAF Brasil"),
)

# Mais Alimentos
df_mais_alimentos_uf = filtrar_uf(df_mais_alimentos_totalizadores, "Mais Alimentos")
for indicador, coluna, formato in [
    ("Contratos", "qtd_contratos", "inteiro"),
    ("Valor total dos contratos", "valor_total_contratos", "moeda"),
]:
    adicionar_indicador(
        linhas_politicas,
        "Mais Alimentos",
        indicador,
        somar_coluna(df_mais_alimentos_uf, coluna, "Mais Alimentos UF"),
        somar_coluna(df_mais_alimentos_totalizadores, coluna, "Mais Alimentos Brasil"),
        formato=formato,
    )

# ATER
df_ater_uf = filtrar_uf(df_ater_totalizadores, "ATER")
for indicador, coluna in [
    ("Famílias com ATER iniciada no ano", "familias_com_ater_iniciada_no_ano"),
    ("Famílias com ATER recebida no ano", "familias_com_ater_recebida_no_ano"),
]:
    adicionar_indicador(
        linhas_politicas,
        "ATER",
        indicador,
        somar_coluna(df_ater_uf, coluna, "ATER UF"),
        somar_coluna(df_ater_totalizadores, coluna, "ATER Brasil"),
    )

# Garantia-Safra
df_garantia_safra_uf = filtrar_uf(df_garantia_safra_totalizadores, "Garantia-Safra")
for indicador, coluna in [
    ("Agricultores aderidos", "agricultores_aderidos"),
    ("Agricultores com pagamento liberado", "agricultores_com_pagamento_liberado"),
]:
    adicionar_indicador(
        linhas_politicas,
        "Garantia-Safra",
        indicador,
        somar_coluna(df_garantia_safra_uf, coluna, "Garantia-Safra UF"),
        somar_coluna(df_garantia_safra_totalizadores, coluna, "Garantia-Safra Brasil"),
    )

# PNCF
df_pncf_uf = filtrar_uf(df_pncf_totalizadores, "PNCF")
pncf_operacoes_uf = somar_coluna(df_pncf_uf, "numero_operacoes", "PNCF UF")
pncf_operacoes_br = somar_coluna(df_pncf_totalizadores, "numero_operacoes", "PNCF Brasil")
pncf_valor_uf = somar_coluna(df_pncf_uf, "valor_liberado", "PNCF UF")
pncf_valor_br = somar_coluna(df_pncf_totalizadores, "valor_liberado", "PNCF Brasil")
adicionar_indicador(linhas_politicas, "PNCF", "Operações", pncf_operacoes_uf, pncf_operacoes_br)
adicionar_indicador(linhas_politicas, "PNCF", "Valor liberado", pncf_valor_uf, pncf_valor_br, formato="moeda")
adicionar_indicador(
    linhas_politicas,
    "PNCF",
    "Valor médio liberado por operação",
    dividir_com_seguranca(pncf_valor_uf, pncf_operacoes_uf),
    dividir_com_seguranca(pncf_valor_br, pncf_operacoes_br),
    formato="moeda",
)

# PNRA
df_pnra_uf = filtrar_uf(df_pnra_totalizadores, "PNRA")
for indicador, coluna in [
    ("Famílias em PA criado diferenciado", "numero_familias_pa_criado_diferenciado_pnra"),
    ("Famílias em PA criado tradicional", "numero_familias_pa_criado_tradicional_pnra"),
    ("Famílias em reconhecimento", "numero_familias_reconhecimento_pnra"),
    ("Famílias em regularização", "numero_familias_regularizacao_pnra"),
    ("Total de famílias", "total_numero_familias_pnra"),
]:
    adicionar_indicador(
        linhas_politicas,
        "PNRA",
        indicador,
        somar_coluna(df_pnra_uf, coluna, "PNRA UF"),
        somar_coluna(df_pnra_totalizadores, coluna, "PNRA Brasil"),
    )


df_politicas_resumo = pd.DataFrame(linhas_politicas)
df_politicas_resumo

,politica,indicador,uf,brasil,percentual_uf_br,formato
0,CAF,CAFs Pessoa Física ativos,2.664780e+05,3.898268e+06,6.835805,inteiro
1,CAF,CAFs Pessoa Jurídica ativos,6.810000e+02,9.161000e+03,7.433686,inteiro
2,CAF,Mulheres em CAF ativo,2.526450e+05,3.529596e+06,7.157901,inteiro
3,CAF,Homens em CAF ativo,2.767060e+05,3.519584e+06,7.861895,inteiro
4,PRONAF,Contratos,4.484800e+04,4.820170e+05,9.304236,inteiro
5,PRONAF,Valor total dos contratos,1.604295e+09,1.467108e+10,10.935082,moeda
6,PRONAF,Operações,4.484800e+04,4.820170e+05,9.304236,inteiro
7,Mais Alimentos,Contratos,9.036000e+03,1.907100e+05,4.738084,inteiro
8,Mais Alimentos,Valor total dos contratos,4.158824e+08,7.066669e+09,5.885126,moeda
9,ATER,Famílias com ATER iniciada no ano,5.800000e+02,5.785000e+03,10.025929,inteiro


## 13. Gerar relatório Word de teste

Esta geração `.docx` separa as políticas públicas em seções próprias, seguindo a lógica editorial do `documento_padrao_v1.docx` e os requisitos do PRD. O arquivo é salvo em `OUTPUT_DIR`, que no Colab aponta para o Google Drive e no modo local aponta para `relatorios_gerados/AAAAMM`.

In [75]:
import re

INFORMACAO_INDISPONIVEL = "Informação não disponível nas bases atuais."

UF_NOMES = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins",
}

POLITICAS_RELATORIO = [
    {
        "nome": "CAF",
        "titulo": "CAF - Cadastro da Agricultura Familiar",
        "descricao": "Indicadores consolidados de CAF ativo, com recorte de pessoa física, pessoa jurídica e gênero.",
        "fonte": "CAF Transparência",
        "df_referencia": df_caf_totalizadores,
        "lacunas": [
            "Jovens em CAF ativo.",
            "Quilombolas, indígenas e povos e comunidades tradicionais com DAP.",
            "Principais produtos da agricultura familiar no estado.",
            "Produtos com maior valor total de produção no estado.",
        ],
    },
    {
        "nome": "PRONAF",
        "titulo": "PRONAF",
        "descricao": "Dados gerais de contratos, operações e volume financeiro do PRONAF.",
        "fonte": "GAIA",
        "df_referencia": df_pronaf_totalizadores,
        "lacunas": ["Série histórica anual detalhada no formato do documento padrão."],
    },
    {
        "nome": "Mais Alimentos",
        "titulo": "Mais Alimentos",
        "descricao": "Quantidade de contratos e valor total dos contratos do Mais Alimentos.",
        "fonte": "GAIA",
        "df_referencia": df_mais_alimentos_totalizadores,
        "lacunas": ["Série histórica anual no formato do documento padrão."],
    },
    {
        "nome": "ATER",
        "titulo": "ATER",
        "descricao": "Famílias com ATER iniciada e recebida no ano.",
        "fonte": "Base ATER/ANATER",
        "df_referencia": df_ater_totalizadores,
        "lacunas": [],
    },
    {
        "nome": "Garantia-Safra",
        "titulo": "Garantia-Safra",
        "descricao": "Agricultores aderidos e agricultores com pagamento liberado no Garantia-Safra.",
        "fonte": "Garantia-Safra",
        "df_referencia": df_garantia_safra_totalizadores,
        "lacunas": ["Seção sem equivalente direto no documento padrão v1; incluída porque há base disponível."],
    },
    {
        "nome": "PNCF",
        "titulo": "PNCF",
        "descricao": "Número de operações, valor liberado e valor médio liberado por operação.",
        "fonte": "PNCF",
        "df_referencia": df_pncf_totalizadores,
        "lacunas": ["Série histórica anual no formato do documento padrão."],
    },
    {
        "nome": "PNRA",
        "titulo": "PNRA - Reforma Agrária",
        "descricao": "Famílias por tipo de assentamento, reconhecimento e regularização no PNRA.",
        "fonte": "PNRA",
        "df_referencia": df_pnra_totalizadores,
        "lacunas": ["Quadros históricos anuais no formato do documento padrão."],
    },
]


def adicionar_cabecalho_tabela(tabela, cabecalhos: list[str]) -> None:
    for idx, titulo in enumerate(cabecalhos):
        cell = tabela.rows[0].cells[idx]
        cell.text = titulo
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True


def formatar_valor_relatorio(valor: float, formato: str) -> str:
    if formato == "moeda":
        return formatar_moeda(valor)
    return formatar_inteiro(valor)


def formatar_data_documento(valor) -> str:
    if valor is None or pd.isna(valor):
        return INFORMACAO_INDISPONIVEL

    texto = str(valor).strip()
    if not texto:
        return INFORMACAO_INDISPONIVEL

    data_ano_mes_dia = re.fullmatch(r"(\d{4})[_-](\d{2})[_-](\d{2})", texto)
    if data_ano_mes_dia:
        ano, mes, dia = data_ano_mes_dia.groups()
        return f"{dia}/{mes}/{ano}"

    data_ano_mes = re.fullmatch(r"(\d{4})[_-](\d{2})", texto)
    if data_ano_mes:
        ano, mes = data_ano_mes.groups()
        return f"{mes}/{ano}"

    data = pd.to_datetime(texto, errors="coerce")
    if not pd.isna(data):
        return data.strftime("%d/%m/%Y")
    return texto


def obter_primeiro_valor(df: pd.DataFrame, coluna: str):
    if coluna not in df.columns:
        return None
    serie = df[coluna].dropna()
    if serie.empty:
        return None
    return serie.iloc[0]


def adicionar_fonte_referencia(doc: Document, politica: dict) -> None:
    referencia = formatar_data_documento(obter_primeiro_valor(politica["df_referencia"], "dt_referencia"))
    geracao = formatar_data_documento(obter_primeiro_valor(politica["df_referencia"], "dt_geracao"))
    doc.add_paragraph(f"Fonte: {politica['fonte']}. Referência: {referencia}. Geração da base: {geracao}.")


def adicionar_lacunas(doc: Document, lacunas: list[str]) -> None:
    if not lacunas:
        return
    doc.add_paragraph("Lacunas em relação ao documento padrão:")
    for lacuna in lacunas:
        doc.add_paragraph(f"{lacuna} {INFORMACAO_INDISPONIVEL}", style="List Bullet")


def adicionar_tabela_indicadores(doc: Document, df_indicadores: pd.DataFrame, uf: str) -> None:
    table = doc.add_table(rows=1, cols=4)
    table.style = "Table Grid"
    adicionar_cabecalho_tabela(table, ["Indicador", "Brasil", uf, "Percentual em relação ao nacional"])

    for _, row in df_indicadores.iterrows():
        cells = table.add_row().cells
        cells[0].text = str(row["indicador"])
        cells[1].text = formatar_valor_relatorio(row["brasil"], row["formato"])
        cells[2].text = formatar_valor_relatorio(row["uf"], row["formato"])
        cells[3].text = formatar_percentual(row["percentual_uf_br"])


doc = Document()

nome_uf = UF_NOMES.get(UF_INTERESSE, UF_INTERESSE)

doc.add_paragraph("Versão 1")
titulo = doc.add_heading("Relatório Estadual de Monitoramento", level=0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
subtitulo = doc.add_paragraph(nome_uf)
subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(f"Gerado em: {RUN_TIMESTAMP.strftime('%d/%m/%Y %H:%M')}")
doc.add_paragraph(
    "Documento preliminar gerado a partir das bases de políticas públicas disponíveis. "
    "A estrutura segue o documento padrão v1 como referência editorial."
)

for numero, politica in enumerate(POLITICAS_RELATORIO, start=1):
    df_secao = df_politicas_resumo[df_politicas_resumo["politica"] == politica["nome"]].copy()

    doc.add_heading(f"{numero}. {politica['titulo']}", level=1)
    doc.add_paragraph(politica["descricao"])

    if df_secao.empty:
        doc.add_paragraph(INFORMACAO_INDISPONIVEL)
    else:
        adicionar_tabela_indicadores(doc, df_secao, UF_INTERESSE)

    adicionar_fonte_referencia(doc, politica)
    adicionar_lacunas(doc, politica["lacunas"])

nome_relatorio = f"relatorio_estadual_monitoramento_{UF_INTERESSE.lower()}_{RUN_DATETIME}.docx"
caminho_relatorio = OUTPUT_DIR / nome_relatorio
doc.save(caminho_relatorio)

print(f"Relatório Word gerado em: {caminho_relatorio}")

Relatório Word gerado em: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\relatorios_gerados\202604\relatorio_estadual_monitoramento_mg_20260424195152.docx
